# M4: Generative Information Extraction (LLM)

**Cíl:** Využít přirozenou generativní sílu LLM k extrakci biasových slov z novinových titulků.  
Na rozdíl od M3 (token-by-token klasifikace) zde model dostane celou větu a má vrátit JSON s detekcí a extrakcí anomálního slova.

**Hypotéza:** LLM s lingvistickou teorií (LJMPNIK) v promptu dosáhne lepších výsledků než LLM s generickým promptem.

**Metoda:**
- Prompt A (Base) — generický popis biasu
- Prompt B (Theory-guided) — LJMPNIK taxonomie s příklady

**Evaluace:** Token-level matching — predikované slovo musí odpovídat ground truth tokenu (exact/substring/lemma match).

## 1. Setup & Import

In [1]:
import sys
import os
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix, classification_report
)

%load_ext autoreload
%autoreload 2
%matplotlib inline

# Přidání src do PYTHONPATH
current_dir = os.getcwd()
src_dir = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_dir not in sys.path:
    sys.path.append(src_dir)

# Import vlastních modulů
import config
import data_splitting
import visualization
import evaluation
from llm_client import LLMClassifier

# Nastavení vizualizačního stylu
visualization.setup_style()

print(f"✅ Setup hotov. Data dir: {config.DATA_DIR}")
print(f"   Results dir: {config.RESULTS_DIR}")

⚙️ Configuration loaded. Device: cpu


2026-03-24 16:25:40,264 - INFO - Visualization style applied (Set2 + ggplot-inspired rcParams).


✅ Setup hotov. Data dir: C:\Users\dobes\Documents\UniversityCodingProject_10-02-26\DiplomaThesis\ThesisCoding\data
   Results dir: C:\Users\dobes\Documents\UniversityCodingProject_10-02-26\DiplomaThesis\ThesisCoding\results


## 2. Načtení testovacích dat

Načteme GOLD token-level data. Potřebujeme:
- `target_sentence` — celý text věty
- Ground truth LJMPNIK tokeny (label=1) pro evaluaci extrakce

In [2]:
# Načtení token-level dat přes data_splitting (test split, bez POS filtru)
data = data_splitting.get_train_val_test_splits(
    scenario='baseline',
    level='token',
    filter_type='none',
    random_state=config.RANDOM_SEED
)

df_test = data['meta_test'].copy()
df_test['true_label'] = data['y_test']

# Rekonstrukce vět z tokenů
sentence_groups = df_test.groupby('sentence_id')
sentence_texts = sentence_groups['form'].apply(lambda x: ' '.join(x)).to_dict()
df_test['sentence_text'] = df_test['sentence_id'].map(sentence_texts)

# Ground truth: pro každou větu zjistíme LJMPNIK tokeny (label=1)
gt_tokens = (
    df_test[df_test['true_label'] == 1]
    .groupby('sentence_id')
    .agg(
        gt_words=('form', list),
        gt_lemmas=('lemma', list)
    )
)

# Sentence-level ground truth label (1 pokud věta obsahuje alespoň 1 LJMPNIK token)
sentence_labels = df_test.groupby('sentence_id').agg(
    sentence_text=('sentence_text', 'first'),
    has_bias=('true_label', 'max')  # 1 pokud jakýkoli token je bias
).reset_index()

# Připojení GT tokenů
sentence_labels = sentence_labels.merge(gt_tokens, on='sentence_id', how='left')
sentence_labels['gt_words'] = sentence_labels['gt_words'].apply(
    lambda x: x if isinstance(x, list) else []
)
sentence_labels['gt_lemmas'] = sentence_labels['gt_lemmas'].apply(
    lambda x: x if isinstance(x, list) else []
)

print(f"📊 Testovací set:")
print(f"   Počet vět: {len(sentence_labels)}")
print(f"   S biasem: {sentence_labels['has_bias'].sum()} ({sentence_labels['has_bias'].mean():.1%})")
print(f"   Neutrální: {(sentence_labels['has_bias'] == 0).sum()}")
print(f"\n📌 Ukázka:")
sentence_labels.head()

2026-03-24 16:25:40,371 - INFO - 📊 Preparing scenario: baseline (token level, none filter)
2026-03-24 16:25:40,755 - INFO - ✅ Loaded 17557 rows from C:\Users\dobes\Documents\UniversityCodingProject_10-02-26\DiplomaThesis\ThesisCoding\data\processed\gold_tokens.pkl
2026-03-24 16:25:42,650 - INFO - ✅ Loaded 78991 rows from C:\Users\dobes\Documents\UniversityCodingProject_10-02-26\DiplomaThesis\ThesisCoding\data\processed\silver_tokens.pkl
2026-03-24 16:25:42,660 - INFO - Splitting 520 documents: 104 test, 41 val, 375 train
2026-03-24 16:25:42,667 - INFO - ✅ Document-level split completed:
2026-03-24 16:25:42,668 - INFO -    Train: 376 docs, 4369 samples
2026-03-24 16:25:42,668 - INFO -    Val:   41 docs, 454 samples
2026-03-24 16:25:42,669 - INFO -    Test:  103 docs, 1246 samples
2026-03-24 16:25:42,670 - INFO -    ✓ No document leakage detected between splits
2026-03-24 16:25:42,671 - INFO - ✅ Scenario data prepared:
2026-03-24 16:25:42,672 - INFO -    Train: 4369 samples (L0: 1470, L1

KeyError: "Column(s) ['lemma'] do not exist"

## 3. Definice System Promptů

Dvě varianty promptu pro testování hypotézy o vlivu lingvistické teorie:

- **Prompt A (Base):** Generický popis biasu bez specifické taxonomie
- **Prompt B (Theory-guided):** LJMPNIK taxonomie s konkrétními kategoriemi a příklady

In [ ]:
# ============================================================
# SYSTEM PROMPTS
# ============================================================

SYSTEM_PROMPT_A = (
    "Jsi editor zpravodajství. Přečti si následující větu. "
    "Rozhodni, zda obsahuje nějaký subjektivní, hodnotící nebo emocionálně zabarvený výraz (bias). "
    "Vrať striktně JSON ve formátu: "
    '{\"bias\": 1, \"word\": \"konkrétní_slovo\"} '
    "nebo "
    '{\"bias\": 0, \"word\": null}. '
    "Nepiš nic jiného než tento JSON."
)

SYSTEM_PROMPT_B = (
    "Jsi expert na mediální lingvistiku. "
    "Hledáme tzv. LJMPNIK (Lexikální jednotky s potenciálem narušit informační kvalitu). Patří sem: "
    "1. Expresiva (např. 'žvanit', 'praštěný'), "
    "2. Hodnotící adjektiva (např. 'skandální', 'absurdní', 'kontroverzní'), "
    "3. Manipulativní adverbia (např. 'údajně', 'prý', 'pochopitelně'), "
    "4. Pejorativa a meliorativa (např. 'zpackat', 'vynikající'), "
    "5. Intenzifikátory (např. 'naprosto', 'zcela', 'totálně'). "
    "Analyzuj větu a vrať striktně JSON: "
    '{\"bias\": 1, \"word\": \"konkrétní_slovo\"} '
    "nebo "
    '{\"bias\": 0, \"word\": null}. '
    "Vrať pouze jedno nejnápadnější slovo. Nepiš nic jiného než JSON."
)

PROMPT_VARIANTS = {
    'A_base': SYSTEM_PROMPT_A,
    'B_theory': SYSTEM_PROMPT_B,
}

print("✅ System prompty definovány:")
for name, prompt in PROMPT_VARIANTS.items():
    print(f"\n--- {name} ---")
    print(prompt[:120] + "...")

✅ System prompty definovány:

--- A_base ---
Jsi editor zpravodajství. Přečti si následující větu. Rozhodni, zda obsahuje nějaký subjektivní, hodnotící nebo emocioná...

--- B_theory ---
Jsi expert na mediální lingvistiku. Hledáme tzv. LJMPNIK (Lexikální jednotky s potenciálem narušit informační kvalitu). ...


## 4. Pomocné funkce: JSON parsing & token matching

Robustní parsování JSON z LLM odpovědí (model nemusí vždy vrátit čistý JSON) a matching extrahovaného slova proti ground truth.

In [ ]:
def parse_llm_json(raw_response: str) -> dict:
    """
    Robustně parsuje JSON z LLM odpovědi.
    Zvládá: čistý JSON, JSON v markdown bloku, text okolo JSONu.
    
    Returns:
        dict s klíči 'bias' (int) a 'word' (str|None), 
        nebo {'bias': None, 'word': None, 'parse_error': True} při selhání.
    """
    if not raw_response or not isinstance(raw_response, str):
        return {'bias': None, 'word': None, 'parse_error': True, 'raw': str(raw_response)}
    
    text = raw_response.strip()
    
    # 1. Pokus: extrahuj JSON z markdown code bloku ```json ... ```
    md_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if md_match:
        text = md_match.group(1)
    
    # 2. Pokus: najdi první {...} v textu
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            bias_val = parsed.get('bias')
            word_val = parsed.get('word')
            
            # Normalizace
            if bias_val in [0, 1, '0', '1', True, False]:
                bias_val = int(bias_val)
            else:
                bias_val = None
                
            if word_val in [None, 'null', 'None', '']:
                word_val = None
            elif isinstance(word_val, str):
                word_val = word_val.strip().lower()
            
            return {'bias': bias_val, 'word': word_val, 'parse_error': False, 'raw': raw_response}
        except json.JSONDecodeError:
            pass
    
    # 3. Fallback: selhání parsování
    return {'bias': None, 'word': None, 'parse_error': True, 'raw': raw_response}


def word_matches_gt(predicted_word: str, gt_words: list, gt_lemmas: list) -> bool:
    """
    Kontroluje, zda predikované slovo odpovídá nějakému ground truth tokenu.
    
    Matching strategie (od nejpřísnější):
    1. Exact match (case-insensitive)
    2. Lemma match (predikované slovo = GT lemma)
    3. Substring match (predikované slovo je podřetězcem GT slova nebo naopak)
    
    Returns:
        True pokud se slovo shoduje s jakýmkoli GT tokenem.
    """
    if not predicted_word or not gt_words:
        return False
    
    pred_lower = predicted_word.lower().strip()
    
    for gt_word, gt_lemma in zip(gt_words, gt_lemmas):
        gt_w = gt_word.lower().strip()
        gt_l = gt_lemma.lower().strip() if gt_lemma else ""
        
        # Exact match
        if pred_lower == gt_w:
            return True
        # Lemma match
        if pred_lower == gt_l:
            return True
        # Substring match (min 3 znaky aby se předešlo falešným shodám)
        if len(pred_lower) >= 3:
            if pred_lower in gt_w or gt_w in pred_lower:
                return True
            if gt_l and (pred_lower in gt_l or gt_l in pred_lower):
                return True
    
    return False


# Rychlý test
assert parse_llm_json('{"bias": 1, "word": "skandální"}')['bias'] == 1
assert parse_llm_json('```json\n{"bias": 0, "word": null}\n```')['word'] is None
assert parse_llm_json('Tady je výsledek: {"bias": 1, "word": "test"}')['word'] == 'test'
assert parse_llm_json('nesmysl bez jsonu')['parse_error'] == True
assert word_matches_gt('skandální', ['skandální'], ['skandální']) == True
assert word_matches_gt('skandáln', ['skandální'], ['skandální']) == True  # substring
assert word_matches_gt('xy', ['skandální'], ['skandální']) == False  # příliš krátký substring
print("✅ Parsovací a matching funkce otestovány.")

## 5. Inicializace LLM klienta

In [ ]:
# Inicializace LLM klienta (Gemini)
llm = LLMClassifier(provider="gemini", model_name="gemini-1.5-flash")
print(f"✅ LLM klient inicializován: {llm.provider} / {llm.model_name}")

# Ověření funkčnosti pomocí jednoduché testovací věty
test_sentence = sentence_labels.iloc[0]['sentence_text']
test_response = llm.generate(
    prompt=f'Věta: "{test_sentence}"',
    system_prompt=SYSTEM_PROMPT_A
)
print(f"\n🧪 Test volání:")
print(f"   Věta: {test_sentence}")
print(f"   Raw odpověď: {test_response}")
print(f"   Parsováno: {parse_llm_json(test_response)}")

## 6. Spuštění experimentu — Generative Extraction

Pro každou větu z testovacího setu:
1. Pošli větu do LLM s Prompt A (Base), ulož výsledek
2. Pošli větu do LLM s Prompt B (Theory-guided), ulož výsledek
3. Parsuj JSON odpovědi a vyhodnoť matching

In [ ]:
def run_m4_experiment(df: pd.DataFrame, llm_client: LLMClassifier, 
                      prompt_variants: dict, sleep_between: float = 0.5) -> pd.DataFrame:
    """
    Spustí M4 Generative Extraction experiment přes všechny věty a prompt varianty.
    
    Args:
        df: DataFrame s 'sentence_text', 'has_bias', 'gt_words', 'gt_lemmas'
        llm_client: Inicializovaný LLMClassifier s metodou generate()
        prompt_variants: Dict {název: system_prompt}
        sleep_between: Pauza mezi voláními (rate limit ochrana)
    
    Returns:
        DataFrame s výsledky pro každou kombinaci (věta × prompt varianta)
    """
    results = []
    total = len(df) * len(prompt_variants)
    
    with tqdm(total=total, desc="M4 Generative Extraction") as pbar:
        for _, row in df.iterrows():
            sentence = row['sentence_text']
            user_prompt = f'Věta: "{sentence}"'
            
            for variant_name, system_prompt in prompt_variants.items():
                # Volání LLM
                raw_response = llm_client.generate(
                    prompt=user_prompt,
                    system_prompt=system_prompt
                )
                
                # Parsování odpovědi
                parsed = parse_llm_json(raw_response)
                
                # Evaluace matching (pro věty s biasem)
                predicted_bias = parsed['bias'] if parsed['bias'] is not None else 0
                predicted_word = parsed['word']
                
                # Word-level match (jen pokud model řekl bias=1 a GT má bias)
                word_match = False
                if predicted_bias == 1 and predicted_word and row['has_bias'] == 1:
                    word_match = word_matches_gt(
                        predicted_word, row['gt_words'], row['gt_lemmas']
                    )
                
                results.append({
                    'sentence_id': row['sentence_id'],
                    'sentence_text': sentence,
                    'gt_label': row['has_bias'],
                    'gt_words': row['gt_words'],
                    'prompt_variant': variant_name,
                    'pred_bias': predicted_bias,
                    'pred_word': predicted_word,
                    'word_match': word_match,
                    'parse_error': parsed['parse_error'],
                    'raw_response': parsed.get('raw', ''),
                })
                
                pbar.update(1)
                time.sleep(sleep_between)
    
    return pd.DataFrame(results)

print("✅ Funkce run_m4_experiment() definována.")

In [ ]:
%%time

# Spuštění experimentu
df_results_m4 = run_m4_experiment(
    df=sentence_labels,
    llm_client=llm,
    prompt_variants=PROMPT_VARIANTS,
    sleep_between=0.5  # Pauza mezi voláními kvůli rate limitu
)

# Základní statistiky
print(f"\n📊 Výsledky experimentu:")
print(f"   Celkem záznamů: {len(df_results_m4)}")
print(f"   Parse errors: {df_results_m4['parse_error'].sum()} ({df_results_m4['parse_error'].mean():.1%})")
print(f"\n   Výsledky per varianta:")
for variant in PROMPT_VARIANTS:
    subset = df_results_m4[df_results_m4['prompt_variant'] == variant]
    print(f"   {variant}: pred_bias=1 v {subset['pred_bias'].mean():.1%}, "
          f"word_match rate={subset['word_match'].mean():.1%}, "
          f"parse_errors={subset['parse_error'].sum()}")

In [ ]:
# Uložení surových výsledků pro pozdější analýzu
results_dir = config.RESULTS_DIR / "m4_generative"
results_dir.mkdir(parents=True, exist_ok=True)

df_results_m4.to_csv(results_dir / "m4_raw_results.csv", index=False)
print(f"💾 Výsledky uloženy: {results_dir / 'm4_raw_results.csv'}")

## 7. Evaluace — Sentence-level detekce & Token-level extrakce

Dvě úrovně evaluace:
1. **Sentence-level (detekce):** Dokáže model správně identifikovat, zda věta obsahuje bias? → standard F1/Precision/Recall
2. **Token-level (extrakce):** Pokud model řekl bias=1, shoduje se extrahované slovo s GT? → extraction accuracy

In [ ]:
def evaluate_m4_variant(df_variant: pd.DataFrame, variant_name: str) -> dict:
    """
    Vyhodnotí jednu prompt variantu na sentence i token úrovni.
    
    Returns:
        Dict s metrikami pro danou variantu.
    """
    y_true = df_variant['gt_label'].values
    y_pred = df_variant['pred_bias'].values
    
    # --- Sentence-level metriky (detekce) ---
    sent_metrics = evaluation.calculate_metrics(y_true, y_pred, y_scores=y_pred.astype(float))
    
    # --- Token-level metriky (extrakce) ---
    # TP = model řekl bias=1 A slovo se shoduje s GT
    # FP = model řekl bias=1 ALE slovo se neshoduje (nebo GT nemá bias)
    # FN = GT má bias ALE model řekl bias=0
    # TN = GT nemá bias A model řekl bias=0
    
    bias_sentences = df_variant[df_variant['gt_label'] == 1]
    neutral_sentences = df_variant[df_variant['gt_label'] == 0]
    
    # Extraction: z vět s biasem, kolik slov model správně identifikoval
    tp_extract = bias_sentences['word_match'].sum()
    fn_extract = len(bias_sentences) - bias_sentences['pred_bias'].sum()  # model řekl 0 pro biasovou větu
    fp_no_match = ((bias_sentences['pred_bias'] == 1) & (~bias_sentences['word_match'])).sum()  # řekl 1 ale špatné slovo
    fp_neutral = (neutral_sentences['pred_bias'] == 1).sum()  # řekl 1 pro neutrální větu
    
    extraction_precision = tp_extract / (tp_extract + fp_no_match + fp_neutral) if (tp_extract + fp_no_match + fp_neutral) > 0 else 0.0
    extraction_recall = tp_extract / len(bias_sentences) if len(bias_sentences) > 0 else 0.0
    extraction_f1 = (2 * extraction_precision * extraction_recall / (extraction_precision + extraction_recall) 
                     if (extraction_precision + extraction_recall) > 0 else 0.0)
    
    result = {
        'variant': variant_name,
        # Sentence-level
        'sent_precision': sent_metrics['precision'],
        'sent_recall': sent_metrics['recall'],
        'sent_f1': sent_metrics['f1'],
        'sent_auprc': sent_metrics['avg_precision'],
        'sent_accuracy': sent_metrics['accuracy'],
        # Token-level extraction
        'ext_precision': extraction_precision,
        'ext_recall': extraction_recall,
        'ext_f1': extraction_f1,
        'ext_tp': int(tp_extract),
        'ext_fp_wrong_word': int(fp_no_match),
        'ext_fp_neutral': int(fp_neutral),
        'ext_fn': int(fn_extract),
        # Metadata
        'parse_errors': int(df_variant['parse_error'].sum()),
        'n_sentences': len(df_variant),
    }
    
    return result


# Evaluace obou variant
metrics_list = []
for variant_name in PROMPT_VARIANTS:
    df_variant = df_results_m4[df_results_m4['prompt_variant'] == variant_name].copy()
    metrics = evaluate_m4_variant(df_variant, variant_name)
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)

print("=" * 70)
print("📊 SENTENCE-LEVEL DETEKCE (má věta bias?)")
print("=" * 70)
print(df_metrics[['variant', 'sent_precision', 'sent_recall', 'sent_f1', 'sent_auprc', 'sent_accuracy']].to_string(index=False))

print("\n" + "=" * 70)
print("🎯 TOKEN-LEVEL EXTRAKCE (správné slovo?)")
print("=" * 70)
print(df_metrics[['variant', 'ext_precision', 'ext_recall', 'ext_f1', 'ext_tp', 'ext_fp_wrong_word', 'ext_fp_neutral', 'ext_fn']].to_string(index=False))

print(f"\n⚠️  Parse errors: {df_metrics[['variant', 'parse_errors']].to_string(index=False)}")

## 8. Bootstrap CI & Permutační test

In [ ]:
# Bootstrap CI pro sentence-level F1
print("📊 Bootstrap 95% CI pro Sentence-level F1:")
print("-" * 50)

for variant_name in PROMPT_VARIANTS:
    df_v = df_results_m4[df_results_m4['prompt_variant'] == variant_name]
    y_true = df_v['gt_label'].values
    y_pred = df_v['pred_bias'].values
    
    lower, upper = evaluation.calculate_bootstrap_ci(
        y_true, y_pred, 
        metric_func=f1_score, 
        n_bootstraps=2000
    )
    point = f1_score(y_true, y_pred)
    print(f"   {variant_name}: F1 = {point:.3f} [{lower:.3f}, {upper:.3f}]")

# Permutační test: je B signifikantně lepší než A?
print("\n📊 Permutační test (B_theory vs. A_base):")
print("-" * 50)
df_a = df_results_m4[df_results_m4['prompt_variant'] == 'A_base'].sort_values('sentence_id')
df_b = df_results_m4[df_results_m4['prompt_variant'] == 'B_theory'].sort_values('sentence_id')

p_value = evaluation.permutation_test(
    y_true=df_a['gt_label'].values,
    y_pred1=df_b['pred_bias'].values,  # H1: B je lepší
    y_pred2=df_a['pred_bias'].values,  # H0: A
    metric_func=f1_score,
    n_permutations=10000
)
print(f"   p-value = {p_value:.4f} {'(signifikantní ✅)' if p_value < 0.05 else '(nesignifikantní)'}")

## 9. Vizualizace — Srovnání prompt variant

### 9.1 Srovnání Sentence-level metrik (detekce)

In [ ]:
# --- Srovnávací barplot: Sentence-level metriky ---
fig, axes = plt.subplots(1, 2, figsize=config.VIZ_CONFIG['figure_sizes']['large'])

variant_labels = {
    'A_base': 'Prompt A\n(Base)',
    'B_theory': 'Prompt B\n(Theory-guided)'
}
colors_variants = [config.COLORS['l0'], config.COLORS['l1']]

# 9.1a — Sentence-level F1, Precision, Recall
metrics_to_plot = ['sent_f1', 'sent_precision', 'sent_recall']
metric_labels = ['F1', 'Přesnost', 'Úplnost']

x = np.arange(len(metrics_to_plot))
width = 0.35

for i, (_, row) in enumerate(df_metrics.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    bars = axes[0].bar(x + i * width, values, width, 
                       label=variant_labels[row['variant']], 
                       color=colors_variants[i], alpha=0.85)
    for bar, val in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

axes[0].set_xticks(x + width / 2)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylabel('Skóre')
axes[0].set_ylim(0, 1.15)
axes[0].legend(loc='upper right')
# axes[0].set_title() — titulky v LaTeX \caption

# 9.1b — AUPRC srovnání
for i, (_, row) in enumerate(df_metrics.iterrows()):
    bar = axes[1].bar(i, row['sent_auprc'], color=colors_variants[i], 
                      alpha=0.85, label=variant_labels[row['variant']])
    axes[1].text(i, row['sent_auprc'] + 0.01, f'{row["sent_auprc"]:.3f}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].set_xticks(range(len(df_metrics)))
axes[1].set_xticklabels([variant_labels[v] for v in df_metrics['variant']])
axes[1].set_ylabel('AUPRC')
axes[1].set_ylim(0, 1.15)
# axes[1].set_title() — titulky v LaTeX \caption

plt.tight_layout()

save_path = results_dir / "m4_sentence_level_comparison.png"
fig.savefig(save_path, dpi=config.VIZ_CONFIG['dpi']['print'], bbox_inches='tight')
print(f"💾 Graf uložen: {save_path}")
plt.show()

### 9.2 Srovnání Token-level extrakce

In [ ]:
# --- Srovnávací barplot: Token-level extrakce ---
fig, ax = plt.subplots(figsize=config.VIZ_CONFIG['figure_sizes']['medium'])

ext_metrics = ['ext_f1', 'ext_precision', 'ext_recall']
ext_labels = ['Extrakční F1', 'Extrakční přesnost', 'Extrakční úplnost']

x = np.arange(len(ext_metrics))
width = 0.35

for i, (_, row) in enumerate(df_metrics.iterrows()):
    values = [row[m] for m in ext_metrics]
    bars = ax.bar(x + i * width, values, width,
                  label=variant_labels[row['variant']],
                  color=colors_variants[i], alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(ext_labels)
ax.set_ylabel('Skóre')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
# ax.set_title() — titulky v LaTeX \caption

plt.tight_layout()

save_path = results_dir / "m4_extraction_comparison.png"
fig.savefig(save_path, dpi=config.VIZ_CONFIG['dpi']['print'], bbox_inches='tight')
print(f"💾 Graf uložen: {save_path}")
plt.show()

### 9.3 Confusion matrix pro obě varianty

In [ ]:
# Confusion matrix pro obě varianty vedle sebe
fig, axes = plt.subplots(1, 2, figsize=config.VIZ_CONFIG['figure_sizes']['large'])

for i, variant_name in enumerate(PROMPT_VARIANTS):
    df_v = df_results_m4[df_results_m4['prompt_variant'] == variant_name]
    y_true = df_v['gt_label'].values
    y_pred = df_v['pred_bias'].values
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Neutrální', 'Bias'], 
                yticklabels=['Neutrální', 'Bias'])
    axes[i].set_xlabel('Predikce')
    axes[i].set_ylabel('Skutečnost')
    axes[i].text(0.5, -0.15, variant_labels[variant_name].replace('\n', ' '),
                 transform=axes[i].transAxes, ha='center', fontsize=11)
    # axes[i].set_title() — titulky v LaTeX \caption

plt.tight_layout()

save_path = results_dir / "m4_confusion_matrices.png"
fig.savefig(save_path, dpi=config.VIZ_CONFIG['dpi']['print'], bbox_inches='tight')
print(f"💾 Graf uložen: {save_path}")
plt.show()

## 10. Error Analysis — Kvalitativní rozbor chyb

In [ ]:
# --- Error Analysis: ukázky kde model selhal / uspěl ---
print("=" * 70)
print("🔍 KVALITATIVNÍ ANALÝZA CHYB (Prompt B — Theory-guided)")
print("=" * 70)

df_b = df_results_m4[df_results_m4['prompt_variant'] == 'B_theory'].copy()

# True Positives se správným slovem
tp_correct = df_b[(df_b['gt_label'] == 1) & (df_b['word_match'] == True)]
print(f"\n✅ True Positive + správné slovo ({len(tp_correct)} případů):")
for _, row in tp_correct.head(5).iterrows():
    print(f"   Věta: \"{row['sentence_text'][:80]}...\"")
    print(f"   → Predikce: '{row['pred_word']}' | GT: {row['gt_words']}")

# False Positives (model řekl bias, ale věta je neutrální)
fp = df_b[(df_b['gt_label'] == 0) & (df_b['pred_bias'] == 1)]
print(f"\n❌ False Positive — model halucunuje bias ({len(fp)} případů):")
for _, row in fp.head(5).iterrows():
    print(f"   Věta: \"{row['sentence_text'][:80]}...\"")
    print(f"   → Predikce: '{row['pred_word']}' (GT: neutrální)")

# False Negatives (model řekl neutrální, ale věta má bias)
fn = df_b[(df_b['gt_label'] == 1) & (df_b['pred_bias'] == 0)]
print(f"\n⚠️  False Negative — model přehlédl bias ({len(fn)} případů):")
for _, row in fn.head(5).iterrows():
    print(f"   Věta: \"{row['sentence_text'][:80]}...\"")
    print(f"   → GT slova: {row['gt_words']}")

# Správná detekce ale špatné slovo
wrong_word = df_b[(df_b['gt_label'] == 1) & (df_b['pred_bias'] == 1) & (~df_b['word_match'])]
print(f"\n🎯 Detekce OK, ale špatné slovo ({len(wrong_word)} případů):")
for _, row in wrong_word.head(5).iterrows():
    print(f"   Věta: \"{row['sentence_text'][:80]}...\"")
    print(f"   → Predikce: '{row['pred_word']}' | GT: {row['gt_words']}")

# Parse errors
parse_err = df_b[df_b['parse_error'] == True]
print(f"\n⚠️  Parse errors ({len(parse_err)} případů):")
for _, row in parse_err.head(3).iterrows():
    print(f"   Raw: \"{str(row['raw_response'])[:100]}...\"")

## 11. Souhrnná tabulka výsledků & Export

In [ ]:
# Souhrnná tabulka pro diplomovou práci
summary = df_metrics[[
    'variant', 'sent_f1', 'sent_precision', 'sent_recall', 'sent_auprc',
    'ext_f1', 'ext_precision', 'ext_recall', 'parse_errors'
]].copy()

summary.columns = [
    'Varianta promptu', 'Detekce F1', 'Detekce Přesnost', 'Detekce Úplnost', 'AUPRC',
    'Extrakce F1', 'Extrakce Přesnost', 'Extrakce Úplnost', 'Parse chyby'
]

# Formátování
for col in summary.columns:
    if col not in ['Varianta promptu', 'Parse chyby']:
        summary[col] = summary[col].apply(lambda x: f"{x:.3f}")

print("📋 Souhrnná tabulka M4 — Generative Information Extraction")
print("=" * 90)
print(summary.to_string(index=False))

# Export do CSV
summary.to_csv(results_dir / "m4_summary_metrics.csv", index=False)
df_metrics.to_csv(results_dir / "m4_detailed_metrics.csv", index=False)
print(f"\n💾 Metriky uloženy do: {results_dir}")

print("\n✅ Notebook M4 dokončen.")